In [21]:
#replace create_dataloader in model.py

import torch
from tokenizer import Tokenizer
from gan_layers import Discriminator
import numpy as np
from evaluation import evaluation
from mol_metrics import *
from torch.utils.data import DataLoader
import pandas as pd


def create_dataloader(data, batch_size=1024, shuffle=True, num_workers=0):
    
    
    def b_tokenize(batch):

        smiles = [item['SMILES'] for item in batch]
        qed_values = [item['QED'] for item in batch]
        solubility_values = [item['Solubility'] for item in batch]
        sa_values = [item['SA'] for item in batch]

        tokenized_smiles = tokenizer.batch_tokenize(smiles)

        return (tokenized_smiles, qed_values, solubility_values, sa_values)

    

    
    return DataLoader(
        data,
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=b_tokenize,
        num_workers=num_workers
    )





    

In [22]:
#replace data in experiment.py
import pandas as pd
from rdkit import Chem
from mol_metrics import *



train_smiles = []
with open('zinc250k.csv', "r") as f:
    for line in f.readlines()[1:]:
        train_smiles.append(line.split(",")[1])

# Training canonical SMILES
data0 = [Chem.MolToSmiles(Chem.MolFromSmiles(sm)) for sm in train_smiles]
data = list(set(data0))

tokenizer = Tokenizer(data)
data = pd.Series(data)
data = data [0:4]


In [23]:
display(type(data))

pandas.core.series.Series

In [24]:
#replace data in experiment.py
import rdkit.Chem as Chem
from rdkit.Chem import PandasTools, QED, Descriptors, rdMolDescriptors

#qed (drug-likeness)
qed =  pd.DataFrame([[data.iloc[i], QED.qed(Chem.MolFromSmiles(data.iloc[i]))] for i in range(len(data))], columns=['SMILES', 'QED'])
#solubility
solubility =  pd.DataFrame([[data.iloc[i], solubility(data.iloc[i])] for i in range(len(data))], columns=['SMILES', 'Solubility'] )
#synthesizability
sa =  pd.DataFrame([[data.iloc[i], SA(data.iloc[i])] for i in range(len(data))], columns=['SMILES', 'SA'])

In [25]:
qed

,SMILES,QED
0,COC(=O)c1cnc(N(C(=O)Cc2ccc(F)cc2F)C2CC2)s1,0.776550
1,CC[C@@H](C)S(=O)(=O)N1CCc2sc(-c3csc(C)n3)cc2C1,0.839766
2,C[NH+](Cc1cccc(OCc2cccc(C#N)c2)c1)C[C@@H]1CCC[...,0.804814
3,Cc1ccc(-n2cnnc2SCN2C(=O)c3ccccc3C2=O)c(C)c1,0.525130


In [26]:
solubility

,SMILES,Solubility
0,COC(=O)c1cnc(N(C(=O)Cc2ccc(F)cc2F)C2CC2)s1,0.620695
1,CC[C@@H](C)S(=O)(=O)N1CCc2sc(-c3csc(C)n3)cc2C1,0.708931
2,C[NH+](Cc1cccc(OCc2cccc(C#N)c2)c1)C[C@@H]1CCC[...,0.543164
3,Cc1ccc(-n2cnnc2SCN2C(=O)c3ccccc3C2=O)c(C)c1,0.655472


In [27]:
sa

,SMILES,SA
0,COC(=O)c1cnc(N(C(=O)Cc2ccc(F)cc2F)C2CC2)s1,0.683893
1,CC[C@@H](C)S(=O)(=O)N1CCc2sc(-c3csc(C)n3)cc2C1,0.477199
2,C[NH+](Cc1cccc(OCc2cccc(C#N)c2)c1)C[C@@H]1CCC[...,0.303142
3,Cc1ccc(-n2cnnc2SCN2C(=O)c3ccccc3C2=O)c(C)c1,0.754328


In [28]:
#replace data in experiment.py

# Combine the data frames into one
combined_data = pd.concat([qed, solubility['Solubility'], sa['SA']], axis=1)

combined_data
data = combined_data.to_dict('records')

In [29]:
data

[{'SMILES': 'COC(=O)c1cnc(N(C(=O)Cc2ccc(F)cc2F)C2CC2)s1',
  'QED': 0.7765496840173333,
  'Solubility': 0.6206954099485521,
  'SA': 0.6838925255333211},
 {'SMILES': 'CC[C@@H](C)S(=O)(=O)N1CCc2sc(-c3csc(C)n3)cc2C1',
  'QED': 0.8397658635331341,
  'Solubility': 0.7089314050200395,
  'SA': 0.47719900648958785},
 {'SMILES': 'C[NH+](Cc1cccc(OCc2cccc(C#N)c2)c1)C[C@@H]1CCC[C@@H]1O',
  'QED': 0.804814257570462,
  'Solubility': 0.543164039914193,
  'SA': 0.30314164380371594},
 {'SMILES': 'Cc1ccc(-n2cnnc2SCN2C(=O)c3ccccc3C2=O)c(C)c1',
  'QED': 0.5251296770175583,
  'Solubility': 0.6554719686790125,
  'SA': 0.7543275411749205}]

In [30]:
#replace loader in experiment.py

loader = create_dataloader(data=data, batch_size=2, shuffle=True, num_workers=0)

In [31]:
tokenized_smiles_list = []
for batch in loader:
    tokenized_smiles, qed_values, solubility_values, sa_values = batch
    print("Tokenized SMILES:", tokenized_smiles)
    print("QED values:", qed_values)
    print("Solubility values:", solubility_values)
    print("SA values:", sa_values)
    print()
    

Tokenized SMILES: tensor([[ 3, 11,  3, 34, 32, 11, 31, 14, 28, 14,  1, 14, 34, 18, 34,  3, 34, 32,
         11, 31,  3, 14, 26, 14, 14, 14, 34,  5, 31, 14, 14, 26,  5, 31,  3, 26,
          3,  3, 26, 31, 12, 28, 35,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0],
        [ 3, 21, 18, 13, 10,  4, 34,  3, 14, 28, 14, 14, 14, 14, 34, 11,  3, 14,
         26, 14, 14, 14, 14, 34,  3, 20, 18, 31, 14, 26, 31, 14, 28, 31,  3, 21,
          3, 16, 16, 13,  4, 28,  3,  3,  3, 21,  3, 16, 16, 13,  4, 28, 11, 35]])
QED values: [0.7765496840173333, 0.804814257570462]
Solubility values: [0.6206954099485521, 0.543164039914193]
SA values: [0.6838925255333211, 0.30314164380371594]

Tokenized SMILES: tensor([[ 3, 14, 28, 14, 14, 14, 34, 19,  1, 26, 14,  1,  1, 14, 26,  6,  3, 18,
         26,  3, 34, 32, 11, 31, 14, 27, 14, 14, 14, 14, 14, 27,  3, 26, 32, 11,
         31, 14, 34,  3, 31, 14, 28, 35,  0,  0,  0],
        [ 3,  3, 21,  3, 16, 16, 13,  4, 34,  3, 31,  6, 34, 32, 11, 31, 34, 32,
         11, 

In [32]:
tokenized_smiles

tensor([[ 3, 14, 28, 14, 14, 14, 34, 19,  1, 26, 14,  1,  1, 14, 26,  6,  3, 18,
         26,  3, 34, 32, 11, 31, 14, 27, 14, 14, 14, 14, 14, 27,  3, 26, 32, 11,
         31, 14, 34,  3, 31, 14, 28, 35,  0,  0,  0],
        [ 3,  3, 21,  3, 16, 16, 13,  4, 34,  3, 31,  6, 34, 32, 11, 31, 34, 32,
         11, 31, 18, 28,  3,  3, 14, 26, 12, 14, 34, 19, 14, 27, 14, 12, 14, 34,
          3, 31,  1, 27, 31, 14, 14, 26,  3, 28, 35]])

In [42]:
tokenized_smiles.size()

torch.Size([2, 47])

In [33]:
type(tokenized_smiles)

torch.Tensor

In [34]:
qed_values

[0.5251296770175583, 0.8397658635331341]

In [35]:
type(qed_values[0])

float

In [38]:
torch.set_printoptions(precision=20)

In [39]:
# Convert qed_values into a 2D tensor that aligns with the shape of tokenized_smiles
qed_values_tensor = torch.tensor(qed_values, dtype=torch.float64).view(-1, 1).expand(-1, tokenized_smiles.shape[1])

print(qed_values_tensor)

tensor([[0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830507, 0.52512967701755830507, 0.52512967701755830507,
         0.52512967701755830

In [43]:
qed_values_tensor.size()

torch.Size([2, 47])

In [44]:
qed_values_tensor.type

<function Tensor.type>

In [ ]:
type(qed_values)

In [ ]:
type(solubility_values)

In [ ]:
solubility_values

In [3]:
iiiii = 0.

In [4]:
iiiii

0.0

AttributeError: 'float' object has no attribute 'type'

In [ ]:
data[0]['SMILES']


In [ ]:
data[1]['SMILES']

In [ ]:

tokenized_smiles_list

In [ ]:
def a(x, y):
    def b(x, y):
        z = x + y
        return z
    z = b(x, y)
    return z



In [ ]:
a(2, 2)

In [ ]:
b = [1,2,3,4,4,4,4,4]

In [ ]:
b

In [ ]:
d = list(set(b))

In [ ]:
d

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import Draw

#generated_smiles = ["CC", "CCC", "C", ""]
generated_smiles = [ ""]

molecules = np.array([Chem.MolFromSmiles(s) for s in generated_smiles if len(s.strip())])

print(molecules.shape)
# Display the molecules
Draw.MolsToGridImage(molecules, subImgSize=(200, 200))

In [ ]:
import pandas as pd
from rdkit import Chem
from mol_metrics import *
import numpy as np
import os



training_smiles = []
with open('zinc250k.csv', "r") as f:
    for line in f.readlines()[1:]:
        training_smiles.append(line.split(",")[1])

# Training canonical SMILES
train_smiles0 = [Chem.MolToSmiles(Chem.MolFromSmiles(sm)) for sm in training_smiles]
train_smiles = list(set(train_smiles0))

In [ ]:
train_smiles = pd.Series(train_smiles)

In [ ]:
train_smiles

In [1]:
decoded_x_gen = ["C1CCCCC1", "C1CCC1", "C1CC1", "", "INVALID"]

In [1]:
decoded_x_gen = ["INVALID"]

In [2]:
from rdkit.Chem import PandasTools, QED, Descriptors, rdMolDescriptors
from mol_metrics import *
from rdkit import Chem
import pandas as pd
import numpy as np

from tokenizer import Tokenizer


tokenizer = Tokenizer(decoded_x_gen)

c:\Users\tang\anaconda3\envs\sac\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
RL_Flag = 'QED'  # Change as needed


generated_mols = np.array([Chem.MolFromSmiles(s) for s in decoded_x_gen if len(s.strip())])


x_gen1 = tokenizer.batch_tokenize(decoded_x_gen)


In [6]:
x_gen1

tensor([[1, 2, 6, 5, 3, 1, 4, 7]])

In [7]:
x_gen1.numel()

8

In [8]:
# Pair encoded and decoded SMILES in tuples
smiles_pairs = list(zip(x_gen1.tolist(), generated_mols))

# Filter out invalid SMILES
valid_smiles_pairs = [(encoded, Chem.MolToSmiles(decoded)) for encoded, decoded in smiles_pairs if decoded != None and decoded.GetNumAtoms() > 1 and Chem.MolToSmiles(decoded) != ' ']

# Unzip the pairs into separate lists
valid_encoded, valid_decoded = zip(*valid_smiles_pairs)

# Convert to DataFrames/Series as needed
valid_encoded_df = pd.Series(valid_encoded)
valid_decoded_df = pd.Series(valid_decoded)





ValueError: not enough values to unpack (expected 2, got 0)

In [ ]:
# QED
if RL_Flag == 'QED':
    generated_reward_df = pd.DataFrame([[valid_encoded_df.iloc[i], valid_decoded_df.iloc[i], QED.qed(Chem.MolFromSmiles(valid_decoded_df.iloc[i]))] for i in range(len(valid_decoded_df))], columns=['Encoded SMILES', 'SMILES', 'QED'])

# Solubility
elif RL_Flag == 'Solubility':
    generated_reward_df = pd.DataFrame([[valid_encoded_df.iloc[i], valid_decoded_df.iloc[i], solubility(valid_decoded_df.iloc[i])] for i in range(len(valid_decoded_df))], columns=['Encoded SMILES', 'SMILES', 'Solubility'])



# SA
elif RL_Flag == 'SA':
    generated_reward_df = pd.DataFrame([[valid_encoded_df.iloc[i], valid_decoded_df.iloc[i], SA(valid_decoded_df.iloc[i])] for i in range(len(valid_decoded_df))], columns=['Encoded SMILES', 'SMILES', 'SA'])


#print('generated_reward_df: ', generated_reward_df)


In [6]:
generated_reward_df

,Encoded SMILES,SMILES,QED
0,"[1, 6, 1, 1, 1, 1, 1, 6, 9]",C1CCCCC1,0.422316
1,"[1, 6, 1, 1, 1, 6, 9, 0, 0]",C1CCC1,0.394810
2,"[1, 6, 1, 1, 6, 9, 0, 0, 0]",C1CC1,0.381425


In [8]:
#convert Encoded SMILES	to 'torch.Tensor'[batch size, seq_len]
import torch
value_column_name = generated_reward_df.columns[-1]
# assuming 'generated_reward_df' is your DataFrame
generated_valid_smiles = torch.tensor(generated_reward_df['Encoded SMILES'].tolist())

#convert QED to 'torch.Tensor'[batch size, seq_len*QED]
generated_data_values = torch.tensor(generated_reward_df[value_column_name].values, dtype=torch.float32).view(-1, 1).expand(-1, generated_valid_smiles.shape[1])



In [12]:
value_column_name

'QED'

In [9]:
generated_valid_smiles

tensor([[1, 6, 1, 1, 1, 1, 1, 6, 9],
        [1, 6, 1, 1, 1, 6, 9, 0, 0],
        [1, 6, 1, 1, 6, 9, 0, 0, 0]])

In [11]:
generated_data_values

tensor([[0.4223, 0.4223, 0.4223, 0.4223, 0.4223, 0.4223, 0.4223, 0.4223, 0.4223],
        [0.3948, 0.3948, 0.3948, 0.3948, 0.3948, 0.3948, 0.3948, 0.3948, 0.3948],
        [0.3814, 0.3814, 0.3814, 0.3814, 0.3814, 0.3814, 0.3814, 0.3814, 0.3814]])

In [ ]:
generated_valid_smiles